# SwarmSort Gen 2, notebook 0: build the merged dataset

**What:** merges the Generation 2 sources into one 7-class YOLO dataset (BIODEGRADABLE, CARDBOARD, GLASS, METAL,
PAPER, PLASTIC, OTHER) with `src/build_dataset.py` and `configs/gen2/sources.yaml`: EXIF-rotated JPEGs with a long
side of at most 640 px (the training size), duplicate-aware 70/20/10 split (Gen 1 test images stay test), and one test set per domain
(studio / real_world / india).

**Why:** every later Gen 2 notebook trains or evaluates on exactly this build. They attach this notebook's output as
input, so the dataset is built once and every arm sees the same images.

**Inputs to attach** (Add Input -> Datasets):

| Dataset | Kaggle slug | Domain | Needed? |
|---|---|---|---|
| TACO | `kneroma/tacotrashdataset` | real_world | yes |
| HITL Recycling | `humansintheloop/recycling-dataset` | real_world | yes |
| Gen 1 garbage detection | `viswaprakash1990/garbage-detection` | studio | yes |
| DWSD (India) | your own private dataset (see `notebooks/gen2/README.md`) | india | optional |

Without DWSD the build still runs (`build_dataset.py` skips that optional source with a loud warning), but there is
no India test set.

**Accelerator: None (CPU only)**, which saves GPU quota; nothing here needs a GPU. **Internet: on** (for the clone).
Run it with **Save Version -> Save & Run All (Commit)**.

**Expected time:** about 15-40 minutes on CPU (reading about 15 GB of inputs and decoding the high-resolution TACO and
HITL photos). No earlier notebook's output is needed.

Output: `/kaggle/working/gen2_data/` (images, labels, lists, `data.yaml`, `data_test_<domain>.yaml`, `split.csv`,
`build_report.json`) and the check grid `/kaggle/working/gen2_check.jpg`.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# No pip install needed: the build uses only numpy, PIL and PyYAML, which Kaggle already has
import numpy, PIL, yaml
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| pillow", PIL.__version__)


def run(*args):
    """Run a repository script. check=True: if the script stops with an error, this cell fails and the version
    is marked failed."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    subprocess.run([sys.executable, *args], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# The sources config: 7 classes and the Kaggle datasets it looks for. build_dataset.py itself stops with the name of
# the dataset to attach if a required source is missing; DWSD is optional (skipped with a loud warning).
sys.path.insert(0, "src")
from build_dataset import load_config
INPUT, SOURCES, OUT = "/kaggle/input", "configs/gen2/sources.yaml", "/kaggle/working/gen2_data"
classes, sources = load_config(SOURCES)
assert len(classes) == 7, f"{SOURCES} has {len(classes)} classes {classes}; Gen 2 uses 7 (the Gen 1 six + OTHER)"
print("Classes:", classes)
for s in sources:
    print(f"  {s['name']:<18} {s['format']:<12} {s['domain']:<11} {'optional' if s['optional'] else 'required':<9} "
          f"attach: {s['kaggle']}")
print("\nAttached under", INPUT)
for root, dirs, _ in os.walk(INPUT):
    depth = root[len(INPUT):].count("/")
    if depth >= 3:
        dirs[:] = []
    print("  " * depth + os.path.basename(root) + "/")
    dirs.sort()


In [ ]:
# The build: about 15-40 minutes on CPU. Re-running it replaces a previous build in the same folder.
run("src/build_dataset.py", "--sources", SOURCES, "--out", OUT, "--seed", 42, "--max-side", 640,
    "--search-root", INPUT)


In [ ]:
# Build report summary: per source, per class and per domain, for each split
import json
import pandas as pd
from IPython.display import display
report = json.load(open(f"{OUT}/build_report.json"))
SPLITS = ["train", "valid", "test"]
per_source = pd.DataFrame({name: {"domain": s["domain"], "found": s["images_found"], "sampled": s["images_sampled"],
                                  "kept": s["images_kept"], **{sp: s["splits"][sp]["images"] for sp in SPLITS}}
                           for name, s in report["sources"].items()}).T
print("Images per source")
display(per_source)
per_class = pd.DataFrame({sp: report["totals"][sp]["boxes_per_class"] for sp in SPLITS})
per_class.loc["total"] = per_class.sum()
print("Boxes per class")
display(per_class)
per_domain = pd.DataFrame({d: {sp: v[sp]["images"] for sp in SPLITS} for d, v in report["domains"].items()}).T
print("Images per domain")
display(per_domain)
for name, s in report["sources"].items():
    if s["images_dropped"] or s["annotations_dropped"]:
        print(f"{name}: images dropped {s['images_dropped']}; annotations dropped {s['annotations_dropped']}")
print("Duplicate groups:", report["duplicates"]["groups"], "| excluded cross-split copies:",
      len(report["duplicates"]["excluded_cross_split_copies"]))
for w in report["warnings"]:
    print("Warning:", w)
for name, why in report.get("skipped_sources", {}).items():
    print(f"*** WARNING: optional source '{name}' is NOT in this build: {why}")


In [ ]:
# Checks: 7 classes, no empty split, and a test set for each domain (india only if DWSD was attached)
assert len(report["classes"]) == 7, f"expected 7 classes, the build has {report['classes']}"
for sp in SPLITS:
    assert report["totals"][sp]["images"] > 0, f"the {sp} split is empty"
variants = report["test_variants"]
for domain in ["studio", "real_world"]:
    assert variants.get(domain, {}).get("test_images", 0) > 0, f"no test images for the {domain} domain"
    assert os.path.isfile(f"{OUT}/data_test_{domain}.yaml"), f"data_test_{domain}.yaml missing"
if variants.get("india", {}).get("test_images", 0) > 0:
    print("india test images:", variants["india"]["test_images"])
else:
    print("WARNING: no india test images (DWSD not attached or not found): g2_6_evaluate will skip india")
print("Test images per domain:", {k: v["test_images"] for k, v in variants.items() if not k.startswith("src_")})
print("Classes:", report["classes"])


In [ ]:
# Check grid: 4 images per domain with their boxes drawn, to eyeball the box alignment
# (TACO photos carry EXIF rotation; a box that misses its object means a rotation problem)
import csv, random
from collections import defaultdict
from PIL import Image, ImageDraw
from IPython.display import Image as ShowImage
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TILE, PER_DOMAIN = 320, 4
rows = [r for r in csv.DictReader(open(f"{OUT}/split.csv", newline="")) if r["dominant_class"] != "background"]
rng = random.Random(0)
picks = []
for domain in sorted({r["domain"] for r in rows}):
    by_source = defaultdict(list)
    for r in sorted(rows, key=lambda r: r["file"]):
        if r["domain"] == domain:
            by_source[r["source"]].append(r)
    for members in by_source.values():
        rng.shuffle(members)
    names, chosen, i = sorted(by_source), [], 0
    while len(chosen) < PER_DOMAIN and any(by_source.values()):  # take turns between the domain's sources
        if by_source[names[i % len(names)]]:
            chosen.append(by_source[names[i % len(names)]].pop())
        i += 1
    picks.append((domain, chosen))
grid = Image.new("RGB", (TILE * PER_DOMAIN, TILE * len(picks)), "white")
for row, (domain, chosen) in enumerate(picks):
    for col, r in enumerate(chosen):
        img = Image.open(f"{OUT}/images/{r['file']}").convert("RGB")
        w, h = img.size
        img.thumbnail((TILE, TILE))
        s = img.width / w
        draw = ImageDraw.Draw(img)
        for line in open(f"{OUT}/labels/{os.path.splitext(r['file'])[0]}.txt").read().split("\n"):
            if line.strip():
                c, cx, cy, bw, bh = line.split()
                c, cx, cy, bw, bh = int(c), float(cx) * w * s, float(cy) * h * s, float(bw) * w * s, float(bh) * h * s
                draw.rectangle([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], outline=COLORS[c % 8], width=2)
                draw.text((cx - bw / 2 + 2, cy - bh / 2 + 1), report["classes"][c], fill="white", stroke_width=2,
                          stroke_fill=COLORS[c % 8])
        draw.text((3, img.height - 12), f"{domain} | {r['source']}", fill="white", stroke_width=2, stroke_fill="black")
        grid.paste(img, (col * TILE + (TILE - img.width) // 2, row * TILE + (TILE - img.height) // 2))
grid.save("/kaggle/working/gen2_check.jpg", quality=90)
print("Saved /kaggle/working/gen2_check.jpg:", ", ".join(f"{d} ({len(c)})" for d, c in picks))
display(ShowImage("/kaggle/working/gen2_check.jpg"))


In [ ]:
# Output size (Kaggle keeps at most about 20 GB of /kaggle/working)
def folder_bytes(path):
    return sum(os.path.getsize(os.path.join(root, f)) for root, _, files in os.walk(path) for f in files)

size = folder_bytes(OUT)
print(f"{OUT}: {size / 1e9:.2f} GB in {len(os.listdir(f'{OUT}/images'))} images")
print(f"/kaggle/working in total: {folder_bytes('/kaggle/working') / 1e9:.2f} GB")
assert size < 19e9, "the build is close to Kaggle's 20 GB output limit: lower --max-side"


### Next: attach this output to every later notebook

Every later Gen 2 notebook (`g2_1_arm_a` to `g2_6_evaluate`) needs this build: in each one, use
**Add Input -> Notebook Output -> `g2_0_build_dataset`** and pick this finished version. They find `gen2_data/`
inside the attached output by themselves and write their own configs that point at it (`build_dataset.py
--relocate-from`), so the images are never copied again. Look at the check grid above first: every box should sit on
its object.

### Files to download / where they go locally

| File in the Output tab | Where it goes locally |
|---|---|
| `gen2_data/build_report.json` | `results/gen2/build_report.json` (the dataset numbers for the report) |
| `gen2_data/split.csv` | `configs/gen2/split.csv` (which image went to which split) |
| `gen2_check.jpg` (optional) | `results/gen2/plots/gen2_check.jpg` |

Commit both small files. The images stay on Kaggle.

### If the session dies

The build is deterministic for the same seed, inputs and settings, so just run it again.